<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Homework 5: Practice a Neural Network Coding Workflow</h1>
</div>

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1)

[Homework Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Homework_index.ipynb)

### ME 5995: AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science

Wayne State University

**Individual graded Homework**

**Version: Student Notebook**

[Open in Colab](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/homework/hw05_neural_network_classification.ipynb)

## Student Information

Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Overview and objectives

Complete a neural-network classification workflow using manufacturing sound.
Build a 16-neuron model and compare it with supplied
32-neuron reference results. All other settings are fixed.

By the end, you can independently complete the core workflow, use validation
checkpoints, compare model size and results, and diagnose one preprocessing error.
This is individual graded Homework. Submit it as instructed in Canvas.
Exact dates and allowed resources are controlled by the assignment instructions.

Use Lab 8/9 and the API reference below as learning references.
This notebook supplies reference results, not a complete model-building solution.
Do not add searches over architectures, seeds or learning rates.

## Required work: 100 points

| Part | Evidence | Points |
|---|---|---:|
| 1 | Training-only standardization | 15 |
| 2 | Correct 16-neuron model and parameter count | 20 |
| 3 | Compile, fit and use the validation checkpoint | 20 |
| 4 | Prediction, class conversion and evaluation | 15 |
| 5 | Quantitative comparison and conclusion | 20 |
| 6 | One preprocessing diagnosis | 10 |

Correct implementation and supported interpretation matter. The smaller model
does not need to win, and no minimum accuracy is required.

## Data and evaluation context

The supplied features represent **800 normal MIMII public 1.0 recordings**:
fan, pump, slider and valve; four machine IDs per type; 50 recordings per type/ID.
Each original recording lasts 10 seconds at 16 kHz, retaining source microphone
Channel 1. The source 0 dB condition contains mixed factory background noise.

Each parent recording gives ten nonoverlapping one-second segments. Each segment
is a 32-by-61 log-mel matrix: 32 mel bands and 61 time frames. The numerical log
power reference is digital, not calibrated dB SPL. Extraction is already complete.

Labels are **0=fan, 1=pump, 2=slider, 3=valve**. Machine-type classification is a
course-derived task, not the original MIMII anomalous-sound benchmark.
Only the log-mel values enter the NN. IDs, labels and partition names are metadata.

The fixed same-ID split has 480 training, 160 validation and 160 test recordings,
or 4,800/1,600/1,600 segments. All ten segments from one parent stay together.
The same ID string in different machine types does not identify one physical machine.
Previously used course recordings are not an untouched external benchmark.

**Software/files:** Colab provides TensorFlow/Keras, NumPy, pandas, Matplotlib
and scikit-learn. The supplied feature folder contains `logmel.npz`,
`recordings.csv`, `baseline32_summary.csv`, `FEATURE_DICTIONARY.md` and `LICENSE_DATA.txt`.
No package installation is part of the assessed work.

## Provided setup: run unchanged

The next two cells import packages and locate the supplied files.
This is technical infrastructure, not code you need to reconstruct.
The first learning section then opens the files and inspects what they contain.
No model is trained in this setup block.

In [ ]:
# PROVIDED: a small, reproducible CPU example and familiar plotting tools.
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display
tf.get_logger().setLevel("ERROR")
tf.config.experimental.enable_op_determinism()

In [ ]:
# PROVIDED: use cached files, or retrieve the supplied course data.
data_base_url = "https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/mimii_logmel800/"
data_folder = Path("mimii_logmel800")
data_folder.mkdir(parents=True, exist_ok=True)
feature_file = data_folder / "logmel.npz"
metadata_file = data_folder / "recordings.csv"

In [ ]:
# PROVIDED: download only missing files. Do not edit this setup cell.
from urllib.request import urlretrieve
if not feature_file.is_file():
    urlretrieve(data_base_url + "logmel.npz", feature_file)
if not metadata_file.is_file():
    urlretrieve(data_base_url + "recordings.csv", metadata_file)
reference_file = data_folder / "baseline32_summary.csv"
if not reference_file.is_file():
    urlretrieve(data_base_url + "baseline32_summary.csv", reference_file)
dictionary_file = data_folder / "FEATURE_DICTIONARY.md"
if not dictionary_file.is_file():
    urlretrieve(data_base_url + "FEATURE_DICTIONARY.md", dictionary_file)
license_file = data_folder / "LICENSE_DATA.txt"
if not license_file.is_file():
    urlretrieve(data_base_url + "LICENSE_DATA.txt", license_file)

## First: open the files and understand the data

An `.npz` file is a container of named NumPy arrays. Opening it does not train or
standardize anything. `archive.files` shows the stored names; square brackets
retrieve one named array. The CSV is a separate recording-level metadata table.
Inspect these structures before applying a split or a transformation.

In [ ]:
# PROVIDED: open the feature container and the recording metadata.
archive = np.load(feature_file)
recordings = pd.read_csv(metadata_file)
print("Arrays stored in the NPZ file:", archive.files)
print("Metadata rows and columns:", recordings.shape)

### 1. Read the named arrays

`X` and `y` describe **segments**. `recording_id` and `machine_id` describe
**parent recordings**. `recording_index` links each segment to a parent.
Keep these names separate: an input row is not an independent recording.

In [ ]:
# PROVIDED: extract one named item at a time.
X = archive["X"]
y = archive["y"]
recording_index = archive["recording_index"]
recording_ids = archive["recording_id"]
recording_machine_ids = archive["machine_id"]
class_names = archive["class_names"]

In [ ]:
# PROVIDED: shape tells how many entries each array contains.
print("X shape and dtype:", X.shape, X.dtype)
print("y shape and dtype:", y.shape, y.dtype)
print("recording_index shape:", recording_index.shape)
print("recording_ids shape:", recording_ids.shape)
print("recording_machine_ids shape:", recording_machine_ids.shape)
print("class_names shape:", class_names.shape)

| Array | Shape | Meaning |
|---|---|---|
| X | (8000, 32, 61) | 8000 segments, 32 mel bands, 61 time frames |
| y | (8000,) | One integer machine-type label per segment |
| recording_index | (8000,) | One parent-recording row number per segment |
| recording_ids | (800,) | One stable identifier per parent recording |
| recording_machine_ids | (800,) | One machine ID per parent recording |
| class_names | (4,) | Meaning of label indices 0, 1, 2 and 3 |

`float32` stores numerical log-mel values. `int64` stores integer labels/indices.
A shape `(8000,)` is a one-dimensional array. In X, `X[0]` selects one 32-by-61
matrix; it does not select one frequency band from every segment.

**Part 1 check A: Read the input structure.**

Run the provided shape cells. Record `X.shape`, `y.shape` and `X[0].shape`, then
identify which dimension counts segments. Use the actual Python outputs;
one tuple per array and one short phrase are sufficient.

**Your response:** TODO.

In [ ]:
# PROVIDED: an integer label is an index into class_names.
label_key = pd.DataFrame()
label_key["Label"] = [0, 1, 2, 3]
label_key["Machine type"] = class_names
display(label_key)

### 2. Inspect the recording table

The table has 800 rows, one per 10-second parent recording. A recording ID
identifies a source file. A machine ID identifies a model/ID group within its
machine type. `partition` is the supplied train/validation/test assignment.
These columns support grouping and interpretation; they are not NN inputs.

In [ ]:
# PROVIDED: show a few useful columns, not all source checksums and file paths.
metadata_columns = ["recording_id", "machine_type", "machine_id", "label", "partition"]
metadata_preview = recordings[metadata_columns]
display(metadata_preview.head())

In [ ]:
# PROVIDED: count parent recordings before looking at segment counts.
recording_types = recordings["machine_type"]
metadata_machine_ids = recordings["machine_id"]
type_id_counts = pd.crosstab(recording_types, metadata_machine_ids)
display(type_id_counts)

### 3. Follow one segment back to its parent

The next example inspects the first stored row only to understand file structure.
It is not model evaluation. `recording_index[0]` is the parent-row number.
Using that number in `recording_ids` finds the source recording. No parsing of
filenames is needed. The corresponding class name comes from y, not the ID.

In [ ]:
# PROVIDED: inspect one segment and its parent using explicit variables.
example_segment = X[0]
example_label = y[0]
example_parent_index = recording_index[0]
example_parent_id = recording_ids[example_parent_index]
example_machine_id = recording_machine_ids[example_parent_index]
example_machine_type = class_names[example_label]
print("One segment shape:", example_segment.shape)
print("Label and machine type:", example_label, example_machine_type)
print("Parent row and recording ID:", example_parent_index, example_parent_id)
print("Parent machine ID:", example_machine_id)

### 4. Check why 800 recordings give 8000 segments

Rows 0 through 9 come from one parent recording. Row 10 begins the next parent.
The short preview below shows this relationship. Looking up an array with
several integer indices retrieves one value for each supplied index.
All ten segments from one parent must remain in the same partition.

In [ ]:
# PROVIDED: inspect only twelve rows, rather than printing the full dataset.
segment_rows = np.arange(12)
parent_indices = recording_index[:12]
parent_ids = recording_ids[parent_indices]
segment_labels = y[:12]
lineage_preview = pd.DataFrame()
lineage_preview["Segment row"] = segment_rows
lineage_preview["Parent row"] = parent_indices
lineage_preview["Recording ID"] = parent_ids
lineage_preview["Label"] = segment_labels
display(lineage_preview)

### Provided grouping: recording first

`recording_index` connects each segment to one parent row in `recordings`.
The following indexing copies the parent's partition to every segment. A mask
is a True/False array that selects rows. Do not create a random segment split.

In [ ]:
# PROVIDED: every segment receives its parent recording's partition.
recording_partitions = recordings["partition"].to_numpy()
segment_partitions = recording_partitions[recording_index]
train_mask = segment_partitions == "train"
validation_mask = segment_partitions == "validation"
test_mask = segment_partitions == "test"

In [ ]:
# PROVIDED: each True/False mask selects the matching segment rows.
X_train = X[train_mask]
y_train = y[train_mask]
X_validation = X[validation_mask]
y_validation = y[validation_mask]
X_test = X[test_mask]
y_test = y[test_mask]
print("Segment counts:", len(y_train), len(y_validation), len(y_test))

### One training-global transformation

Compute one mean and one population standard deviation across **all elements of
X_train**. Apply the same pair everywhere. This keeps segment-to-segment level
and contrast differences. It does not make every segment mean 0/SD 1, and it
does not force a Gaussian distribution. `mean()` and `std()` without an axis
combine every element of the supplied array.

## Part 1: Prepare the inputs

Complete the global transformation. Reuse the same training statistics.

**API syntax hint:**

```python
values.mean()
values.std()
scaled_inputs = (inputs - fitted_mean) / fitted_std
```

In [ ]:
# Fit preprocessing using training values; reuse it on held-out inputs.
# TODO: use only X_train to compute the two scalar statistics.
training_mean = ...
training_std = ...
X_train_scaled = ...
X_validation_scaled = ...
X_test_scaled = ...

In [ ]:
# PROVIDED: standardized training statistics should be close to 0 and 1.
print("Training mean and SD used:", training_mean, training_std)
print("Standardized training mean:", X_train_scaled.mean())
print("Standardized training SD:", X_train_scaled.std())
print("Shape after scaling:", X_train_scaled.shape)

In [ ]:
# PROVIDED: obtain the shape evidence for the short response.
print("Training/validation/test input shapes:", X_train_scaled.shape, X_validation_scaled.shape, X_test_scaled.shape)
print("Fitted mean/SD shapes:", training_mean.shape, training_std.shape)

**Part 1 check B: Inspect the fitted transformation.**

Record the scaled training, validation and test shapes and the mean/SD shapes.
State whether standardization changes the number of segments or coordinates.
Use the existing outputs; do not fit another transformation.

**Your response:** TODO.

## Supplied API reference

These are syntax patterns, not a complete solution to this dataset.

```python
values.mean()
values.std()
tf.keras.Sequential([layer_1, layer_2, ...])
tf.keras.layers.Input(shape=(rows, columns))
tf.keras.layers.Flatten()
tf.keras.layers.Dense(units, activation="activation_name")
tf.keras.optimizers.Adam(learning_rate=value)
model.compile(optimizer=optimizer_object, loss="loss_name", metrics=["accuracy"])
model.fit(inputs, labels, validation_data=(val_inputs, val_labels),
      epochs=number, batch_size=number, callbacks=[checkpoint], verbose=2)
model.predict(inputs, verbose=0)
np.argmax(probability_array, axis=axis_number)
accuracy_score(true_labels, predicted_labels)
```

Required conditions: one hidden layer with **16 ReLU units**, four softmax outputs,
integer labels, sparse categorical cross-entropy, Adam 0.001, seed 9, batch 32,
40 epochs and training-global scaling. Keep the recording manifest fixed.
`...` in an executable task cell is a placeholder to replace, not working code.

## Supplied 32-neuron reference results

The reference uses the same data, split, scalar statistics, seed and training
conditions. It selects weights by minimum validation loss. Its test results
are comparison evidence, not a target to beat. Do not tune your settings after
looking at the test. One initialization does not establish general superiority.

In [ ]:
# PROVIDED: read a one-row result table, without nested JSON lookups.
reference_file = data_folder / "baseline32_summary.csv"
reference = pd.read_csv(reference_file)
display(reference)

## Part 2: Construct the model

Create model with the required Input, Flatten and two Dense layers. The hidden layer has 16 units; the output still has four classes.

**API syntax hint:**

```python
tf.keras.Sequential([layer_1, layer_2, ...])
tf.keras.layers.Input(shape=(rows, columns))
tf.keras.layers.Flatten()
tf.keras.layers.Dense(units, activation="activation_name")
```

In [ ]:
# Define the layers for one input; Keras supplies the batch dimension.
# Clear framework state before constructing a fresh model.
tf.keras.backend.clear_session()
# Keep initialization reproducible for the controlled comparison.
tf.keras.utils.set_random_seed(9)
model = ...

In [ ]:
model.summary()

In [ ]:
# PROVIDED: None denotes a variable batch size, not a missing feature.
print("Model input shape:", model.input_shape)
print("Model output shape:", model.output_shape)

**Part 2 check C:** Record the model input/output shapes.
Calculate both Dense-layer parameter counts and their total. Why does the number
of output classes stay unchanged? Include these items in one compact response.

**Your response:** TODO.

## Part 3: Compile and train

Complete the optimizer and loss using the required settings.

**API syntax hint:**

```python
optimizer = tf.keras.optimizers.Adam(learning_rate=rate)
model.compile(optimizer=optimizer, loss="loss_name", metrics=["accuracy"])
```

In [ ]:
# Configure learning and reported metrics; compile does not update weights.
optimizer = ...
model.compile(optimizer=optimizer, loss=..., metrics=["accuracy"])

In [ ]:
# PROVIDED: store the weights with the lowest validation loss.
hw05_candidate_file = Path("hw05_candidate.weights.h5")
hw05_candidate_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    hw05_candidate_file, monitor="val_loss", mode="min",
    save_best_only=True, save_weights_only=True, verbose=0
)

Complete the fit call. Keep 40 epochs, batch 32 and the supplied validation checkpoint.

**API syntax hint:**

```python
history = model.fit(
    inputs, targets, validation_data=(validation_inputs, validation_targets),
    epochs=number_of_epochs, batch_size=batch_size, verbose=2
)
```

`verbose=2` prints one summary per epoch, including training and validation metrics. This changes the display, not the learning settings.

In [ ]:
# Update weights on training data; validation monitors held-out performance.
history = model.fit(
    ..., ..., validation_data=...,
    epochs=40, batch_size=32, shuffle=True,
    callbacks=[hw05_candidate_checkpoint], verbose=2
)

The Keras history stores named lists from training.
Read each list into a variable before plotting it. The plots below use epoch
numbers 1 through 40; Python list positions still start at zero.

In [ ]:
# PROVIDED: retrieve each numerical series separately.
training_losses = history.history["loss"]
validation_losses = history.history["val_loss"]
training_accuracies = history.history["accuracy"]
validation_accuracies = history.history["val_accuracy"]
epoch_numbers = np.arange(1, 41)

In [ ]:
# PROVIDED: an explicit loss plot, without a plotting helper or loop.
plt.figure(figsize=(7, 3))
plt.plot(epoch_numbers, training_losses, label="Training")
plt.plot(epoch_numbers, validation_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy")
plt.title("16-neuron candidate: loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PROVIDED: accuracy is a different metric from loss.
plt.figure(figsize=(7, 3))
plt.plot(epoch_numbers, training_accuracies, label="Training")
plt.plot(epoch_numbers, validation_accuracies, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Segment accuracy")
plt.title("16-neuron candidate: accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PROVIDED: find the saved minimum-validation checkpoint without retraining.
best_position = np.argmin(validation_losses)
best_epoch = int(best_position) + 1
best_validation_loss = validation_losses[best_position]
print("Selected epoch:", best_epoch)
print("Minimum validation loss:", best_validation_loss)
model.load_weights(hw05_candidate_file)

## Part 4: Predict and evaluate

Predict standardized test inputs, then convert each four-probability row to one class index.

**API syntax hint:**

```python
probabilities = model.predict(inputs, verbose=0)
predicted_labels = np.argmax(probabilities, axis=class_axis)
```

In [ ]:
# Obtain probabilities before converting them to class labels.
test_probabilities = ...
test_predictions = ...

In [ ]:
# PROVIDED: inspect one probability vector without adding another fit.
print("Test probability shape:", test_probabilities.shape)
print("Test predicted-label shape:", test_predictions.shape)
print("First probability-row sum:", test_probabilities[0].sum())

**Part 4 check D: Interpret predictions.**

Record both prediction-array shapes and the first probability-row sum.
Explain why `argmax(..., axis=1)` returns one class per segment. A rounded row
sum and one short explanation are sufficient; do not tune the model.

**Your response:** TODO.

Compute segment accuracy using true test labels and your predicted class indices.

**API syntax hint:**

```python
accuracy_score(true_labels, predicted_labels)
```

In [ ]:
# Score predictions against labels from the same partition.
test_segment_accuracy = ...

In [ ]:
# PROVIDED: explicitly group ten segments per parent before averaging.
number_of_test_recordings = len(y_test) // 10
test_probability_groups = test_probabilities.reshape(number_of_test_recordings, 10, 4)
test_label_groups = y_test.reshape(number_of_test_recordings, 10)

In [ ]:
# PROVIDED: compute parent-level probabilities, labels and accuracy in steps.
test_recording_probabilities = test_probability_groups.mean(axis=1)
test_recording_labels = test_label_groups[:, 0]
test_recording_predictions = np.argmax(test_recording_probabilities, axis=1)
test_recording_accuracy = accuracy_score(test_recording_labels, test_recording_predictions)
print("Test segment accuracy:", test_segment_accuracy)
print("Test recording accuracy:", test_recording_accuracy)
recording_confusion = confusion_matrix(test_recording_labels, test_recording_predictions, labels=[0, 1, 2, 3])
recording_display = ConfusionMatrixDisplay(recording_confusion, display_labels=class_names)
recording_display.plot()
plt.title("16-neuron model: actual rows, predicted columns")
plt.tight_layout()
plt.show()

In [ ]:
# PROVIDED: .loc selects the named column in row 0 of the reference table.
reference_parameters = reference.loc[0, "Parameters"]
reference_epoch = reference.loc[0, "Best epoch"]
reference_validation_loss = reference.loc[0, "Minimum validation loss"]
reference_test_accuracy = reference.loc[0, "Test recording accuracy"]
candidate_parameters = model.count_params()

In [ ]:
# PROVIDED: assemble the same comparison explicitly, one column at a time.
comparison = pd.DataFrame()
comparison["Hidden units"] = [32, 16]
comparison["Parameters"] = [reference_parameters, candidate_parameters]
comparison["Best epoch"] = [reference_epoch, best_epoch]
comparison["Minimum validation loss"] = [reference_validation_loss, best_validation_loss]
comparison["Test recording accuracy"] = [reference_test_accuracy, test_recording_accuracy]
display(comparison)
final_training_loss = training_losses[-1]
final_validation_loss = validation_losses[-1]
print("Final training loss:", final_training_loss)
print("Final validation loss:", final_validation_loss)

## Part 5: Compare and conclude

Use at most four short sentences: (1) cite the parameter counts and both
minimum validation losses; (2) compare recording test accuracy; (3) cite one
train/validation-curve observation; (4) state whether the smaller model is a
reasonable choice for this run and one limitation of that conclusion.

**Your response:** TODO.

## Part 6: Diagnose one preprocessing error

A classmate writes `training_mean = X.mean()` and `training_std = X.std()` before
using the supplied masks. Here X contains all 8,000 segments. Explain the issue
in one sentence and state which array should replace X in both expressions.

**Your response:** TODO.

## Optional challenge: inspect a probability vector

Inspect one correctly predicted validation/test segment and, if available, one
incorrectly predicted segment. Which two classes have the highest probabilities?
If all predictions are correct, compare two correct examples instead.
No extra model training, submission or points are required.

## Submission checklist

Rendered notebook checkboxes are not clickable. To record completion, edit this
Markdown cell and change `[ ]` to `[x]`.

- [ ] Enter your name and WSU AccessID.
- [ ] Complete required code and short written responses.
- [ ] Keep the required model/history/metric outputs visible.
- [ ] Restart the runtime and run cells from top to bottom after completing TODOs.
- [ ] Save `HW05_Firstname_Lastname.ipynb` and its matching PDF.
- [ ] Submit both files through Canvas as instructed. Do not submit through GitHub.

No separate report is required. Optional work is ungraded. PDF rendering issues
receive corrective feedback; the notebook is used to assess obscured evidence.

## References and attribution

- [MIMII public 1.0](https://zenodo.org/records/3384388), DOI 10.5281/zenodo.3384388.
- Purohit et al., [MIMII dataset paper](https://arxiv.org/abs/1909.09347), 2019.
- [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).
- [Keras Sequential guide](https://keras.io/guides/sequential_model/).

Original authors: Harsh Purohit, Ryo Tanabe, Kenji Ichige, Takashi Endo,
Yuki Nikaido, Kaori Suefusa and Yohei Kawaguchi (Hitachi, Ltd.).
Course changes: selected normal 0-dB recordings, retained Channel 1, segmented
each recording into ten one-second inputs, computed log-mel features, and added
labels, metadata and a recording-level split. Preserve attribution and ShareAlike
terms when redistributing derived data. See supplied `FEATURE_DICTIONARY.md` and
`LICENSE_DATA.txt`. This is a teaching task, not an official MIMII benchmark.